# Ontology Mapping Review Notebook: MeSH + SNOMED CT

This notebook supports a local review workflow for OWL ontologies:

1. Enter the OWL filename from the current folder
2. Generate candidate mappings to **MeSH** and **SNOMED CT**
3. Review the generated CSV in the same folder
4. Enter the approved CSV filename from the same folder
5. Generate an updated OWL file with approved mappings
6. Use the generated OWL file from the same folder for re-import into WebProtégé

## Notes
- One ontology class can have **multiple external mappings**
- MeSH and SNOMED CT are handled **independently**
- Mapping properties stay blank unless a corresponding match is found
- You may change mapping properties per row during review


In [1]:
%pip install -q rdflib pandas requests openpyxl ipywidgets

Note: you may need to restart the kernel to use updated packages.


## 1) Enter your local OWL filename

In [2]:

from pathlib import Path

WORK_DIR = Path.cwd()
owl_filename = "test.owl"
owl_path = WORK_DIR / owl_filename
if not owl_path.exists():
    raise FileNotFoundError(owl_path)
print("Using OWL:", owl_path)


Using OWL: /Users/pengyins/PyProject/HEROP/sdohplace-data-ontology/relationship_import/test.owl


## 2) Generate candidate mappings to MeSH and SNOMED CT

In [3]:

import re
import time
import requests
import pandas as pd
from rdflib import Graph
from rdflib.namespace import RDF, RDFS, OWL

INPUT_OWL = str(owl_path)
OUTPUT_CSV = str(WORK_DIR / f"{owl_path.stem}_mapping_candidates_mesh_snomed.csv")

MESH_SPARQL = "https://id.nlm.nih.gov/mesh/sparql"
SNOMED_SYSTEM = "http://snomed.info/sct"
SNOMED_EXPAND = "https://snowstorm-training.snomedtools.org/snowstorm/snomed-ct/fhir/ValueSet/$expand"

def normalize_label(text: str) -> str:
    text = text.strip().lower()
    text = text.replace("-", " ")
    text = text.replace("_", " ")
    text = re.sub(r"\s+", " ", text)
    return text

def mesh_exact_label_match(label: str):
    norm = normalize_label(label)
    safe_label = norm.replace('"', '\\"')

    query = f"""
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX meshv: <http://id.nlm.nih.gov/mesh/vocab#>

    SELECT ?d ?lab WHERE {{
      ?d a meshv:Descriptor .
      ?d rdfs:label ?lab .
      FILTER(LCASE(STR(?lab)) = "{safe_label}")
    }}
    LIMIT 1
    """

    resp = requests.get(
        MESH_SPARQL,
        params={"query": query, "format": "JSON", "inference": "true"},
        headers={"Accept": "application/sparql-results+json"},
        timeout=30,
    )
    resp.raise_for_status()
    data = resp.json()
    bindings = data.get("results", {}).get("bindings", [])
    if not bindings:
        return None, None, None

    uri = bindings[0]["d"]["value"]
    lab = bindings[0]["lab"]["value"]
    mesh_id = uri.rstrip("/").split("/")[-1]
    return uri, lab, f"MESH:{mesh_id}"

def snomed_exact_label_match(label: str):
    norm = normalize_label(label)
    try:
        resp = requests.get(
            SNOMED_EXPAND,
            params={
                "url": f"{SNOMED_SYSTEM}?fhir_vs",
                "filter": norm,
                "count": 20,
                "includeDesignations": "true"
            },
            headers={"Accept": "application/fhir+json, application/json"},
            timeout=30,
        )
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        return None, None, None, f"SNOMED lookup error: {e}"

    items = data.get("expansion", {}).get("contains", [])
    for item in items:
        concept_id = item.get("code")
        if not concept_id:
            continue

        candidate_terms = []
        display = item.get("display")
        if display:
            candidate_terms.append(display)

        for designation in item.get("designation", []):
            term = designation.get("value")
            if term:
                candidate_terms.append(term)

        seen = set()
        deduped_terms = []
        for term in candidate_terms:
            if term not in seen:
                seen.add(term)
                deduped_terms.append(term)

        for term in deduped_terms:
            if normalize_label(term) == norm:
                return f"http://snomed.info/id/{concept_id}", term, f"SNOMEDCT:{concept_id}", ""

    return None, None, None, ""

g = Graph()
g.parse(INPUT_OWL)

classes_to_process = []
for cls in g.subjects(RDF.type, OWL.Class):
    labels = list(g.objects(cls, RDFS.label))
    if not labels:
        continue

    label = str(labels[0]).strip()
    if not label:
        continue

    classes_to_process.append((cls, label))

print(f"Loaded ontology: {INPUT_OWL}")
print(f"Classes with labels to process: {len(classes_to_process)}")

rows = []
mesh_matches = 0
snomed_matches = 0
start_time = time.time()

for index, (cls, label) in enumerate(classes_to_process, start=1):
    print(f"[{index}/{len(classes_to_process)}] Processing: {label}")

    row = {
        "class_iri": str(cls),
        "label": label,
        "normalized_label": normalize_label(label),

        "mesh_uri": "",
        "mesh_label": "",
        "mesh_id": "",
        "mesh_mapping_property": "",
        "approve_mesh": "",

        "snomed_uri": "",
        "snomed_label": "",
        "snomed_id": "",
        "snomed_mapping_property": "",
        "approve_snomed": "",

        "notes": ""
    }

    notes = []

    try:
        mesh_uri, mesh_label, mesh_id = mesh_exact_label_match(label)
        time.sleep(0.15)

        if mesh_uri:
            row["mesh_uri"] = mesh_uri
            row["mesh_label"] = mesh_label
            row["mesh_id"] = mesh_id
            row["mesh_mapping_property"] = "skos:exactMatch"
            mesh_matches += 1
    except Exception as e:
        notes.append(f"MeSH lookup error: {e}")

    snomed_uri, snomed_label, snomed_id, snomed_note = snomed_exact_label_match(label)
    time.sleep(0.15)

    if snomed_uri:
        row["snomed_uri"] = snomed_uri
        row["snomed_label"] = snomed_label
        row["snomed_id"] = snomed_id
        row["snomed_mapping_property"] = "skos:exactMatch"
        snomed_matches += 1
    if snomed_note:
        notes.append(snomed_note)

    if not row["mesh_uri"] and not row["snomed_uri"] and not notes:
        notes.append("No exact MeSH or SNOMED CT match found.")

    row["notes"] = " | ".join(notes)
    rows.append(row)

    mesh_status = "yes" if row["mesh_uri"] else "no"
    snomed_status = "yes" if row["snomed_uri"] else "no"
    print(f"    MeSH: {mesh_status} | SNOMED CT: {snomed_status}")
    if row["notes"]:
        print(f"    Notes: {row['notes']}")
    if index % 10 == 0 or index == len(classes_to_process):
        elapsed = time.time() - start_time
        print(f"Completed {index}/{len(classes_to_process)} classes in {elapsed:.1f}s | MeSH matches: {mesh_matches} | SNOMED matches: {snomed_matches}")

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)

elapsed = time.time() - start_time
print(f"Wrote {OUTPUT_CSV}")
print(f"Rows: {len(df)}")
print(f"Finished in {elapsed:.1f}s | MeSH matches: {mesh_matches} | SNOMED matches: {snomed_matches}")
df.head(50)


Loaded ontology: /Users/pengyins/PyProject/HEROP/sdohplace-data-ontology/relationship_import/test.owl
Classes with labels to process: 101
[1/101] Processing: gender-and-sexuality-dynamics
    MeSH: no | SNOMED CT: no
    Notes: No exact MeSH or SNOMED CT match found.
[2/101] Processing: age-dynamics
    MeSH: no | SNOMED CT: no
    Notes: No exact MeSH or SNOMED CT match found.
[3/101] Processing: education
    MeSH: yes | SNOMED CT: yes
[4/101] Processing: food-insecurity
    MeSH: yes | SNOMED CT: yes
[5/101] Processing: pharmacy-access
    MeSH: no | SNOMED CT: no
    Notes: No exact MeSH or SNOMED CT match found.
[6/101] Processing: soup-kitchen
    MeSH: no | SNOMED CT: no
    Notes: No exact MeSH or SNOMED CT match found.
[7/101] Processing: educational-attainment
    MeSH: no | SNOMED CT: no
    Notes: No exact MeSH or SNOMED CT match found.
[8/101] Processing: body-weight
    MeSH: yes | SNOMED CT: yes
[9/101] Processing: access-to-healthy-foods
    MeSH: yes | SNOMED CT: no
[1

,class_iri,label,normalized_label,mesh_uri,mesh_label,mesh_id,mesh_mapping_property,approve_mesh,snomed_uri,snomed_label,snomed_id,snomed_mapping_property,approve_snomed,notes
0,http://webprotege.stanford.edu/R4Pn614SXJWoqv4...,gender-and-sexuality-dynamics,gender and sexuality dynamics,,,,,,,,,,,No exact MeSH or SNOMED CT match found.
1,http://webprotege.stanford.edu/R4UoHRI6nBVayvF...,age-dynamics,age dynamics,,,,,,,,,,,No exact MeSH or SNOMED CT match found.
2,http://webprotege.stanford.edu/R5HorAzSPd8fjQr...,education,education,http://id.nlm.nih.gov/mesh/2017/D004493,Education,MESH:D004493,skos:exactMatch,,http://snomed.info/id/409073007,Education,SNOMEDCT:409073007,skos:exactMatch,,
3,http://webprotege.stanford.edu/R7L7fXNPV0Vzh3w...,food-insecurity,food insecurity,http://id.nlm.nih.gov/mesh/D000084884,Food Insecurity,MESH:D000084884,skos:exactMatch,,http://snomed.info/id/733423003,Food insecurity,SNOMEDCT:733423003,skos:exactMatch,,
4,http://webprotege.stanford.edu/R7Mq5ZPckHJan7b...,pharmacy-access,pharmacy access,,,,,,,,,,,No exact MeSH or SNOMED CT match found.
5,http://webprotege.stanford.edu/R7NFkmNs27zctNW...,soup-kitchen,soup kitchen,,,,,,,,,,,No exact MeSH or SNOMED CT match found.
6,http://webprotege.stanford.edu/R7QkfbxRCoOlPgQ...,educational-attainment,educational attainment,,,,,,,,,,,No exact MeSH or SNOMED CT match found.
7,http://webprotege.stanford.edu/R7WVU9Wla0XByaT...,body-weight,body weight,http://id.nlm.nih.gov/mesh/2016/D001835,Body Weight,MESH:D001835,skos:exactMatch,,http://snomed.info/id/27113001,Body weight,SNOMEDCT:27113001,skos:exactMatch,,
8,http://webprotege.stanford.edu/R7aLk0jxr5AjqG5...,access-to-healthy-foods,access to healthy foods,http://id.nlm.nih.gov/mesh/D000091483,Access to Healthy Foods,MESH:D000091483,skos:exactMatch,,,,,,,
9,http://webprotege.stanford.edu/R7cnZwDiqLQJTGJ...,social-and-community-context,social and community context,,,,,,,,,,,No exact MeSH or SNOMED CT match found.


## 3) Review the candidate CSV in the same folder

Review the CSV in Excel or Google Sheets.

Recommended values:
- `approve_mesh`: leave blank or enter `yes` to approve, enter `no` to reject
- `approve_snomed`: leave blank or enter `yes` to approve, enter `no` to reject
- `mesh_mapping_property`: `skos:exactMatch`, `skos:closeMatch`, or `owl:equivalentClass`
- `snomed_mapping_property`: `skos:exactMatch`, `skos:closeMatch`, or `owl:equivalentClass`

You may approve **both** mappings for the same class.


In [4]:

print("Candidate CSV written to:", OUTPUT_CSV)


Candidate CSV written to: /Users/pengyins/PyProject/HEROP/sdohplace-data-ontology/relationship_import/test_mapping_candidates_mesh_snomed.csv


## 4) Enter the approved CSV filename

In [5]:

approved_csv_filename = Path(OUTPUT_CSV).name
approved_csv_path = WORK_DIR / approved_csv_filename
if not approved_csv_path.exists():
    raise FileNotFoundError(approved_csv_path)
print("Using approved CSV:", approved_csv_path)


Using approved CSV: /Users/pengyins/PyProject/HEROP/sdohplace-data-ontology/relationship_import/test_mapping_candidates_mesh_snomed.csv


## 5) Generate an updated OWL file from the approved CSV

In [8]:

import pandas as pd
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import OWL, SKOS

INPUT_OWL = str(owl_path)
APPROVED_CSV = str(approved_csv_path)
OUTPUT_OWL = str(WORK_DIR / f"{owl_path.stem}_with_approved_mappings.owl")

OBOINOWL = Namespace("http://www.geneontology.org/formats/oboInOwl#")

predicate_map = {
    "skos:exactMatch": SKOS.exactMatch,
    "skos:closeMatch": SKOS.closeMatch,
    "owl:equivalentClass": OWL.equivalentClass,
}

def is_approved(value) -> bool:
    return str(value).strip().lower() not in {"no", "n", "false", "0"}

g = Graph()
g.parse(INPUT_OWL)

df = pd.read_csv(APPROVED_CSV).fillna("")

added = 0
skipped = 0

for _, row in df.iterrows():
    class_iri = str(row.get("class_iri", "")).strip()
    if not class_iri:
        skipped += 1
        continue

    subj = URIRef(class_iri)

    # MeSH
    if is_approved(row.get("approve_mesh", "")):
        mesh_uri = str(row.get("mesh_uri", "")).strip()
        mesh_id = str(row.get("mesh_id", "")).strip()
        mesh_pred_name = str(row.get("mesh_mapping_property", "")).strip()

        if mesh_uri and mesh_pred_name in predicate_map:
            g.add((subj, predicate_map[mesh_pred_name], URIRef(mesh_uri)))
            if mesh_id:
                g.add((subj, OBOINOWL.hasDbXref, Literal(mesh_id)))
            added += 1
        else:
            skipped += 1

    # SNOMED CT
    if is_approved(row.get("approve_snomed", "")):
        snomed_uri = str(row.get("snomed_uri", "")).strip()
        snomed_id = str(row.get("snomed_id", "")).strip()
        snomed_pred_name = str(row.get("snomed_mapping_property", "")).strip()

        if snomed_uri and snomed_pred_name in predicate_map:
            g.add((subj, predicate_map[snomed_pred_name], URIRef(snomed_uri)))
            if snomed_id:
                g.add((subj, OBOINOWL.hasDbXref, Literal(snomed_id)))
            added += 1
        else:
            skipped += 1

g.serialize(destination=OUTPUT_OWL, format="xml")
print(f"Added mapping assertions: {added}")
print(f"Skipped attempted assertions: {skipped}")
print(f"Wrote {OUTPUT_OWL}")


Added mapping assertions: 44
Skipped attempted assertions: 158
Wrote /Users/pengyins/PyProject/HEROP/sdohplace-data-ontology/relationship_import/test_with_approved_mappings.owl


## 6) Check the generated OWL file in the same folder

In [9]:

print("Updated OWL written to:", OUTPUT_OWL)


Updated OWL written to: /Users/pengyins/PyProject/HEROP/sdohplace-data-ontology/relationship_import/test_with_approved_mappings.owl
